In [3]:
import pandas as pd
from sklearn.ensemble import IsolationForest
import pickle

logon = pd.read_csv('logon.csv')
device = pd.read_csv('device.csv')

print(logon.shape, device.shape)
print(logon.columns.tolist())
print(device.columns.tolist())

(3530285, 5) (1551828, 6)
['id', 'date', 'user', 'pc', 'activity']
['id', 'date', 'user', 'pc', 'file_tree', 'activity']


In [4]:
logon['date'] = pd.to_datetime(logon['date'])
device['date'] = pd.to_datetime(device['date'])

logon['hour'] = logon['date'].dt.hour
logon['day'] = logon['date'].dt.date
device['day'] = device['date'].dt.date

print(logon['activity'].value_counts())

activity
Logon     1948933
Logoff    1581352
Name: count, dtype: int64


In [5]:
logon_events = logon[logon['activity'] == 'Logon'].copy()
logoff_events = logon[logon['activity'] == 'Logoff'].copy()

first_logon = logon_events.groupby(['user', 'pc', 'day']).agg(
    date_in=('date', 'min'),
    hour=('hour', 'first')
).reset_index()

last_logoff = logoff_events.groupby(['user', 'pc', 'day']).agg(
    date_out=('date', 'max')
).reset_index()

sessions = pd.merge(first_logon, last_logoff, on=['user', 'pc', 'day'])
sessions['duration'] = (sessions['date_out'] - sessions['date_in']).dt.total_seconds() / 60
sessions = sessions[sessions['duration'] > 0]

print(sessions.shape)
print(sessions.head())

(1563875, 7)
      user       pc         day             date_in  hour            date_out  \
0  AAB0162  PC-6599  2010-01-04 2010-01-04 07:41:00     7 2010-01-04 18:46:00   
1  AAB0162  PC-6599  2010-01-05 2010-01-05 07:46:00     7 2010-01-05 18:40:00   
2  AAB0162  PC-6599  2010-01-06 2010-01-06 07:45:00     7 2010-01-06 18:55:00   
3  AAB0162  PC-6599  2010-01-07 2010-01-07 07:45:00     7 2010-01-07 18:43:00   
4  AAB0162  PC-6599  2010-01-08 2010-01-08 07:50:00     7 2010-01-08 18:41:00   

   duration  
0     665.0  
1     654.0  
2     670.0  
3     658.0  
4     651.0  


In [6]:
session_features = sessions.groupby(['user', 'day']).agg(
    avg_hour=('hour', 'mean'),
    total_duration=('duration', 'sum'),
    unique_pcs=('pc', 'nunique')
).reset_index()

device_features = device.groupby(['user', 'day']).agg(
    device_connects=('activity', lambda x: (x == 'Connect').sum())
).reset_index()

features_df = pd.merge(session_features, device_features, on=['user', 'day'], how='left')
features_df['device_connects'] = features_df['device_connects'].fillna(0)

print(features_df.shape)
print(features_df.head())

(1393128, 6)
      user         day  avg_hour  total_duration  unique_pcs  device_connects
0  AAB0162  2010-01-04       7.0           665.0           1              0.0
1  AAB0162  2010-01-05       7.0           654.0           1              0.0
2  AAB0162  2010-01-06       7.0           670.0           1              0.0
3  AAB0162  2010-01-07       7.0           658.0           1              0.0
4  AAB0162  2010-01-08       7.0           651.0           1              0.0


In [7]:
feature_cols = ['avg_hour', 'total_duration', 'unique_pcs', 'device_connects']
X = features_df[feature_cols].dropna()

model = IsolationForest(contamination=0.05, random_state=42)
model.fit(X)

preds = model.predict(X)
features_df = features_df.loc[X.index].copy()
features_df['anomaly'] = [1 if p == -1 else 0 for p in preds]

anomalies = features_df[features_df['anomaly'] == 1]
print(f"Total anomalies flagged: {len(anomalies)}")
print(anomalies[['user', 'day', 'avg_hour', 'total_duration', 'unique_pcs', 'device_connects']].head(10))

Total anomalies flagged: 69610
         user         day  avg_hour  total_duration  unique_pcs  \
736   AAC0610  2010-02-06       0.0      250.416667           1   
758   AAC0610  2010-03-09       0.0      989.583333           1   
802   AAC0610  2010-05-11       1.0      913.466667           1   
808   AAC0610  2010-05-19       0.0     1004.216667           1   
872   AAC0610  2010-08-19       2.0      872.600000           1   
915   AAC0610  2010-10-20       0.0     1015.433333           1   
916   AAC0610  2010-10-21       4.0      767.783333           1   
937   AAC0610  2010-11-19       3.0      776.500000           1   
979   AAC0610  2011-01-25       0.0     1002.833333           1   
1001  AAC0610  2011-02-24       0.0     1014.366667           1   

      device_connects  
736               0.0  
758               0.0  
802               2.0  
808               2.0  
872               1.0  
915               1.0  
916               2.0  
937               2.0  
979            

In [8]:
with open('gap1_model_real.pkl', 'wb') as f:
    pickle.dump(model, f)

print("Gap 1 complete. Model saved as gap1_model_real.pkl")
print(f"Total records: {len(features_df)}")
print(f"Anomalies: {len(anomalies)} ({len(anomalies)/len(features_df)*100:.2f}%)")

Gap 1 complete. Model saved as gap1_model_real.pkl
Total records: 1393128
Anomalies: 69610 (5.00%)
